In [0]:
from pyspark.sql.functions import current_timestamp, lit
caminho = "/Volumes/workspace/default/lh_nautical"
tabela = [
    "addresses", "attributes", "brands", "categories", "customers", "employees", "fiscal_invoices", "goods_receipt_items", "goods_receipts", "locations", "order_items", "orders", "payments", "product_suppliers", "product_variants", "products", "purchase_order_items", "purchase_orders", "return_items", "returns", "stock_levels", "suppliers"
]
print("Inciando a c da camada bronze...\n")
for tabela in tabela:
    caminho_csv = f"{caminho}/{tabela}.csv"
    df_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(caminho_csv)
    #acdiona timestamp de ingestão
    df_bronze = df_raw.withColumn("_ingested_at", current_timestamp())
    #escreve no delta
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"default.bronze_{tabela}")
print(f"✅ Tabela 'bronze_{tabela}' criada com {df_bronze.count():,} registros.")

print("\n Camada Bronze finalizada com sucesso!")

Inciando a c da camada bronze...

✅ Tabela 'bronze_suppliers' criada com 25 registros.

🚀 Camada Bronze finalizada com sucesso!


In [0]:
from pyspark.sql.functions import col, count, when, isnan

tabelas = [
    "addresses", "attributes", "brands", "categories", "customers", "employees",
    "fiscal_invoices", "goods_receipt_items", "goods_receipts", "locations",
    "order_items", "orders", "payments", "product_suppliers", "product_variants",
    "products", "purchase_order_items", "purchase_orders", "return_items",
    "returns", "stock_levels", "suppliers"
]

print("--- QUESTÃO 1: EDA - VOLUMETRIA DAS TABELAS BRONZE ---\n")

resumo_eda = []

for t in tabelas:
    df = spark.table(f"default.bronze_{t}")
    n_linhas = df.count()
    n_colunas = len(df.columns)
    resumo_eda.append((t, n_linhas, n_colunas))

df_resumo = spark.createDataFrame(resumo_eda, ["tabela", "n_linhas", "n_colunas"])
df_resumo.orderBy(col("n_linhas").desc()).show(22, truncate=False)

# Checagem de nulos nas tabelas centrais do negócio (as mais usadas nas questões seguintes)
tabelas_criticas = ["customers", "orders", "order_items", "products", "product_variants"]

print("--- NULOS NAS TABELAS CRÍTICAS ---\n")
for t in tabelas_criticas:
    df = spark.table(f"default.bronze_{t}")
    print(f"\nTabela: {t}")
    exprs = [
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]
    df.select(exprs).show(truncate=False)

--- QUESTÃO 1: EDA - VOLUMETRIA DAS TABELAS BRONZE ---

+--------------------+--------+---------+
|tabela              |n_linhas|n_colunas|
+--------------------+--------+---------+
|order_items         |147320  |9        |
|payments            |53546   |10       |
|orders              |48998   |14       |
|fiscal_invoices     |34365   |12       |
|purchase_order_items|6059    |7        |
|stock_levels        |6054    |6        |
|goods_receipt_items |4733    |5        |
|addresses           |3998    |13       |
|customers           |2000    |12       |
|purchase_orders     |2000    |14       |
|goods_receipts      |1548    |7        |
|product_suppliers   |1520    |9        |
|return_items        |1384    |8        |
|product_variants    |1009    |13       |
|returns             |980     |11       |
|products            |500     |11       |
|suppliers           |25      |13       |
|employees           |15      |12       |
|categories          |14      |8        |
|brands             

In [0]:
print("--- QUESTÃO 2: SCHEMA DAS TABELAS BRONZE ---\n")

for t in tabelas:
    df = spark.table(f"default.bronze_{t}")
    print(f"\n=== Tabela: bronze_{t} ===")
    df.printSchema()

--- QUESTÃO 2: SCHEMA DAS TABELAS BRONZE ---


=== Tabela: bronze_addresses ===
root
 |-- id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- address_type: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- street: string (nullable = true)
 |-- number: integer (nullable = true)
 |-- complement: string (nullable = true)
 |-- district: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- is_primary: boolean (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)


=== Tabela: bronze_attributes ===
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- data_type: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)


=== Tabela: bronze_brands ===
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- is_active: boolean (nullable = true)

In [0]:
from pyspark.sql.functions import col, to_timestamp

print("Iniciando o processamento da Camada Silver...\n")
df_orders = spark.table("default.bronze_orders") \
    .withColumn("placed_at", to_timestamp(col("placed_at"))) \
    .withColumn("created_at", to_timestamp(col("created_at"))) \
    .filter(col("status").isin("paid", "completed", "shipped"))
df_orders.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.silver_orders")
print("✅ Tabela 'silver_orders' processada (apenas pedidos válidos).")
# 2. Tratamento de Itens de Pedidos 
df_order_items = spark.table("default.bronze_order_items") \
    .withColumn("unit_price", col("unit_price").cast("double")) \
    .withColumn("quantity", col("quantity").cast("double")) \
    .withColumn("line_total", col("line_total").cast("double"))
df_order_items.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.silver_order_items")
print("✅ Tabela 'silver_order_items' processada.")
# 3. Tratamento de Devoluções C
df_returns = spark.table("default.bronze_returns") \
    .filter(col("status") == "completed")
df_returns.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.silver_returns")
print("✅ Tabela 'silver_returns' processada (apenas devoluções concluídas).")

print("\n Camada Silver finalizada")

Iniciando o processamento da Camada Silver...

✅ Tabela 'silver_orders' processada (apenas pedidos válidos).
✅ Tabela 'silver_order_items' processada.
✅ Tabela 'silver_returns' processada (apenas devoluções concluídas).

🚀 Camada Silver finalizada com sucesso!


In [0]:
#Questões 4 e 5
from pyspark.sql.functions import col, sum as _sum, avg, countDistinct, date_format, expr, to_date, dayofweek

print("Iniciando o processamento da Camada Gold...\n")

# Aliases para facilitar os joins
o = spark.table("default.silver_orders").alias("o")
oi = spark.table("default.silver_order_items").alias("oi")
pv = spark.table("default.bronze_product_variants").alias("pv")
p = spark.table("default.bronze_products").alias("p")
c = spark.table("default.bronze_customers").alias("c")

# Base de vendas
df_vendas_completa = o \
    .join(oi, col("o.id") == col("oi.order_id")) \
    .join(pv, col("oi.product_variant_id") == col("pv.id")) \
    .join(p, col("pv.product_id") == col("p.id"))

# QUESTÃO 4: Clientes Fiéis (Ticket Médio e Diversidade >= 13)
from pyspark.sql.functions import col, sum as _sum, avg, countDistinct, count, expr, to_date, dayofweek

#Faturamento e frequência: direto de 'orders'
df_faturamento = spark.table("default.silver_orders") \
    .groupBy(col("customer_id")) \
    .agg(
        _sum("total").alias("faturamento_total"),
        countDistinct("id").alias("frequencia")
    )

#Diversidade de categorias:
df_diversidade = df_vendas_completa \
    .groupBy(col("o.customer_id").alias("customer_id")) \
    .agg(countDistinct("p.category_id").alias("diversidade_categorias"))

#Junta as duas bases, calcula ticket médio e aplica o filtro
df_q4 = df_faturamento.join(df_diversidade, "customer_id") \
    .withColumn("ticket_medio", col("faturamento_total") / col("frequencia")) \
    .filter(col("diversidade_categorias") >= 13) \
    .join(c, col("customer_id") == col("c.id")) \
    .select("c.id", "c.legal_name", "ticket_medio", "diversidade_categorias", "faturamento_total", "frequencia") \
    .orderBy(col("ticket_medio").desc())

df_q4.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.gold_q4_clientes_fies")

print("--- QUESTÃO 4: TOP 10 CLIENTES FIÉIS (DIVERSIDADE >= 13) ---")
df_q4.show(10, truncate=False)

# QUESTÃO 5: Calendário e Dias em Português (Date Spine)
df_calendario = spark.sql("SELECT explode(sequence(to_date('2020-01-01'), to_date('2026-12-31'), interval 1 day)) as data_completa")

df_vendas_diarias = spark.table("default.silver_orders") \
    .withColumn("data_completa", to_date(col("placed_at"))) \
    .groupBy("data_completa") \
    .agg(_sum("total").alias("venda_do_dia"))

# Mapeamento com dayofweek
df_q5 = df_calendario.join(df_vendas_diarias, "data_completa", "left") \
    .na.fill({"venda_do_dia": 0}) \
    .withColumn("num_dia", dayofweek(col("data_completa"))) \
    .withColumn("dia_semana_pt", expr("""
        CASE num_dia 
            WHEN 1 THEN '1. Domingo'
            WHEN 2 THEN '2. Segunda-feira'
            WHEN 3 THEN '3. Terça-feira'
            WHEN 4 THEN '4. Quarta-feira'
            WHEN 5 THEN '5. Quinta-feira'
            WHEN 6 THEN '6. Sexta-feira'
            WHEN 7 THEN '7. Sábado'
        END
    """)) \
    .groupBy("num_dia", "dia_semana_pt") \
    .agg(avg("venda_do_dia").alias("media_venda_diaria")) \
    .orderBy("num_dia")

df_q5.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("default.gold_q5_vendas_calendario")

print("--- QUESTÃO 5: MÉDIA DE VENDAS POR DIA DA SEMANA (EM PORTUGUÊS) ---")
df_q5.select("dia_semana_pt", "media_venda_diaria").show(7, truncate=False)

print("\n Camada Gold e Relatórios das Questões concluídos com sucesso!")

Iniciando o processamento da Camada Gold...

--- QUESTÃO 4: TOP 10 CLIENTES FIÉIS (DIVERSIDADE >= 13) ---
+----+------------------------+-----------------+----------------------+------------------+----------+
|id  |legal_name              |ticket_medio     |diversidade_categorias|faturamento_total |frequencia|
+----+------------------------+-----------------+----------------------+------------------+----------+
|1527|Caio Farias             |46646.97666666667|13                    |699704.65         |15        |
|1581|Novaes Andrade S.A.     |44501.89749999999|14                    |534022.7699999999 |12        |
|1558|Natália Borges          |44348.15090909092|13                    |487829.6600000001 |11        |
|262 |Castro Cirino S.A.      |43821.1775       |14                    |525854.13         |12        |
|22  |Isadora Rios            |43180.09166666667|14                    |777241.65         |18        |
|1470|Bárbara Albuquerque     |42917.24952380952|14                   

In [0]:
#QUESTÃO 3.2: Soma das Linhas das Tabelas Solicitadas
tabelas_validacao = ["customers", "orders", "order_items", "payments"]
total_linhas = sum([spark.table(f"default.bronze_{t}").count() for t in tabelas_validacao])

print("--- QUESTÃO 3.2: TOTAL DE LINHAS ---")
print(f"Soma total das linhas (customers + orders + order_items + payments): {total_linhas:,}")

--- QUESTÃO 3.2: TOTAL DE LINHAS ---
Soma total das linhas (customers + orders + order_items + payments): 251,864


In [0]:
# QUESTÃO 6: Previsão de Demanda para 'Bússola de Bordo 702'
import pandas as pd
from sklearn.metrics import mean_absolute_error

#tabelas
df_orders = spark.table("default.silver_orders").toPandas()
df_items = spark.table("default.silver_order_items").toPandas()
df_vars = spark.table("default.bronze_product_variants").toPandas()
df_prods = spark.table("default.bronze_products").toPandas()

#Filtrar o produto
bussola_id = df_prods[df_prods['name'].str.contains('Bússola de Bordo 702', case=False, na=False)]['id'].values[0]
vars_bussola = df_vars[df_vars['product_id'] == bussola_id]['id'].tolist()

df_vendas = df_orders.merge(df_items[df_items['product_variant_id'].isin(vars_bussola)], left_on='id', right_on='order_id')
df_vendas['placed_at'] = pd.to_datetime(df_vendas['placed_at'])

#Agrupamento mensal para preenche meses sem venda com 0
ts_mensal = df_vendas.set_index('placed_at').resample('MS')['quantity'].sum().fillna(0)

#Split treino/teste conforme premissas do enunciado
treino = ts_mensal[:'2025-12-31']
teste = ts_mensal['2026-01-01':'2026-03-31']

# 5. BASELINE: média móvel dos últimos 3 meses (sem usar dados futuros)
#    Para cada mês do Q1/2026, a previsão é a média dos 3 meses anteriores já conhecidos.
historico = treino.copy()
previsoes_baseline = []

for data_alvo in teste.index:
    media_3m = historico[-3:].mean()
    previsoes_baseline.append(media_3m)
    # "avança" o histórico incluindo o valor real do mês, só depois de já ter previsto
    historico = pd.concat([historico, pd.Series([teste[data_alvo]], index=[data_alvo])])

previsao_baseline = pd.Series(previsoes_baseline, index=teste.index)
soma_previsao_baseline = int(round(previsao_baseline.sum()))
mae_baseline = mean_absolute_error(teste, previsao_baseline)

print("--- QUESTÃO 6: BASELINE (MÉDIA MÓVEL 3 MESES) ---")
print(f"Previsão mensal (baseline): {previsao_baseline.round(2).to_dict()}")
print(f"Soma total arredondada Q1/2026 (baseline): {soma_previsao_baseline} unidades")
print(f"MAE do baseline: {mae_baseline:.2f}")

# 6. Comparação opcional: SARIMAX
from statsmodels.tsa.statespace.sarimax import SARIMAX

modelo = SARIMAX(treino, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12))
modelo_fit = modelo.fit(disp=False)
previsao_sarimax = modelo_fit.forecast(steps=3)
soma_previsao_sarimax = int(round(previsao_sarimax.sum()))
mae_sarimax = mean_absolute_error(teste, previsao_sarimax)

print("\n--- COMPARAÇÃO: SARIMAX ---")
print(f"Soma total arredondada Q1/2026 (SARIMAX): {soma_previsao_sarimax} unidades")
print(f"MAE do SARIMAX: {mae_sarimax:.2f}")

--- QUESTÃO 6: BASELINE (MÉDIA MÓVEL 3 MESES) ---
Previsão mensal (baseline): {Timestamp('2026-01-01 00:00:00'): 18.67, Timestamp('2026-02-01 00:00:00'): 30.33, Timestamp('2026-03-01 00:00:00'): 30.0}
Soma total arredondada Q1/2026 (baseline): 79 unidades
MAE do baseline: 14.22

--- COMPARAÇÃO: SARIMAX ---
Soma total arredondada Q1/2026 (SARIMAX): 95 unidades
MAE do SARIMAX: 19.21


In [0]:
#QUESTÃO 7: Sistema de Recomendação (Motor de Popa 1949)
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. Unir vendas com produtos
df_vendas_all = df_orders.merge(df_items, left_on='id', right_on='order_id').merge(df_vars, left_on='product_variant_id', right_on='id')

# 2. Matriz Usuário x Produto (1 comprou, 0 não comprou)
matriz_user_prod = pd.crosstab(df_vendas_all['customer_id'], df_vendas_all['product_id']).clip(upper=1)

# 3. Similaridade de Cosseno entre Produtos
matriz_sim = cosine_similarity(matriz_user_prod.T)
df_sim = pd.DataFrame(matriz_sim, index=matriz_user_prod.columns, columns=matriz_user_prod.columns)

# 4. Produto alvo: 'Motor de Popa 1949'
motor_id = df_prods[df_prods['name'].str.contains('Motor de Popa 1949', case=False, na=False)]['id'].values[0]

# Ranking de similaridade
rec_id = df_sim[motor_id].drop(motor_id).idxmax()
rec_nome = df_prods[df_prods['id'] == rec_id]['name'].values[0]
score = df_sim.loc[motor_id, rec_id]

print("--- QUESTÃO 7: RECOMENDAÇÃO DE PRODUTO ---")
print(f"Produto mais similar ao 'Motor de Popa 1949': {rec_nome} (Score Cosseno: {score:.4f})")

--- QUESTÃO 7: RECOMENDAÇÃO DE PRODUTO ---
Produto mais similar ao 'Motor de Popa 1949': Vela Mestra 1913 (Score Cosseno: 0.2046)
